Adapted from https://github.com/dargueso/EHF/

**calcehf_v2.ipynb**
- Calculates daily EHF for land cells in BARRA-R2.
- D'Argueso's EHF code using Nairn and Fawcett (2013) definition of EHF.
- Parallelized using Dask and Xarray (v2 with fixed EHF logic).

**Reads:** "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/1hr/tas/latest/" \
**Writes:** '/scratch/ng72/ms5578/ehf_netcdf/' (populates July–June yearly files) \
**Environment:** analysis3 \
**Compute:** xxlarge


In [ ]:
import sys
sys.path.append('.')

import os
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')

import netCDF4 as nc
import xarray as xr
import numpy as np
import glob as glob
import pandas as pd

import pdb
from itertools import groupby
from Constants import const
from pathlib import Path
import datetime as dt

import dask
import dask.array as da
from dask.distributed import LocalCluster, Client

tas_path should lead to the BARRA-R2 data and write_path is where you want to save files. I've written the code to save yearly files from July-Jun, so it won't cut any heatwaves in half. There was some customization to exclude winters and stuff in the Argueso code, but I don't think it's been preserved when I parallelized it.

In [ ]:
tas_path = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/1hr/tas/latest/"
write_path = '/scratch/ng72/ms5578/ehf_netcdf/'

In [ ]:
cluster = LocalCluster(n_workers=28, threads_per_worker=1, memory_limit="9GB")
client = Client(cluster)
client

I've always used a temperature threshold file, so the actual calculation is untested. Because I haven't parallelized it, it's probably very slow.

In [ ]:
def calc_percentile(tave, nyears, thres_file=None, method="NF13", nwindow=15):
    """Calculate percentile thresholds that identify hot days.

    Parameters
    ----------
    tave : array-like or xarray.DataArray
        Mean daily temperature (time, lat, lon). Can be Dask-backed.
    nyears : int
        Number of years in the analysed period.
    thres_file : str or None
        NetCDF file with precomputed percentiles. When provided, this path is
        used and no computation is done.
    method : {"NF13", "PA13"}
        Method for percentile calculation (Nairn & Fawcett 2013 or Perkins et al. 2013).
    nwindow : int
        Window length (days) for PA13 moving-percentile method.

    Returns
    -------
    pct_calc : array-like or xarray.DataArray
        Percentile thresholds. For NF13: (lat, lon). For PA13: (365, lat, lon).
    """
    # NF13: time-invariant percentile over the base period
    if method == "NF13":
        if thres_file is None:
            print("No thresholds file provided, we will calculate them (NF13)")
            
            # Dask/xarray-optimised path when tave is an xarray.DataArray
            import xarray as xr
            if isinstance(tave, xr.DataArray):
                # Assume first dimension is time (as in this notebook)
                time_dim = tave.dims[0]
                pct_da = tave.quantile(0.95, dim=time_dim, skipna=True)
                # Drop the quantile dimension if present
                if "quantile" in pct_da.dims:
                    pct_da = pct_da.squeeze("quantile", drop=True)
                pct_calc = pct_da
            else:
                # Fallback NumPy path (no Dask)
                if not isinstance(tave, np.ma.core.MaskedArray):
                    pct_calc = np.nanpercentile(tave, 95, axis=0)
                else:
                    pct_calc = np.ones(tave.shape[1:], float) * const.missingval
                    for i in range(tave.shape[1]):
                        for j in range(tave.shape[2]):
                            aux = tave[:, i, j]
                            if len(aux[~aux.mask].data) != 0:
                                pct_calc[i, j] = np.nanpercentile(
                                    aux[~aux.mask].data, 95, axis=0
                                )
        else:
            print("Percentiles are retrieved from the thfile provided (NF13)")
            pct_file = nc.Dataset(thres_file, "r")
            pct_calc = pct_file.variables["PRCTILE95"][:].astype("float")
    
    # PA13: day-of-year dependent percentile (kept as NumPy-style implementation)
    elif method == "PA13":
        if thres_file is None:
            # No percentile file is provided and thus they are calculated from the given data
            print("Percentiles are calculated because no thfile is provided (PA13)")
            windowrange = np.zeros((365,), dtype=bool)
            windowrange[: int(np.ceil(nwindow / 2))] = True
            windowrange[-int(np.floor(nwindow / 2)) :] = True
            if np.sum(windowrange) != nwindow:
                raise SystemExit(0)
            windowrange = np.tile(windowrange, nyears)
            pct_calc = np.ones((365,) + tave.shape[1:], float) * const.missingval
            
            # For PA13, the reference implementation is inherently sequential in time;
            # we retain the original NumPy-style loops for fidelity. If tave is
            # a DataArray, operate on its .values here.
            import xarray as xr
            t_arr = tave.values if isinstance(tave, xr.DataArray) else tave
            
            if not isinstance(t_arr, np.ma.core.MaskedArray):
                for d in range(365):
                    pct_calc[d, :, :] = np.percentile(
                        t_arr[windowrange == True, :, :], 90, axis=0
                    )
                    windowrange = np.roll(windowrange, 1)
            else:
                for i in range(t_arr.shape[1]):
                    for j in range(t_arr.shape[2]):
                        for d in range(365):
                            aux = t_arr[windowrange == True, :, :]
                            if len(aux[~aux.mask].data) != 0:
                                pct_calc[d, :, :] = np.percentile(
                                    aux[~aux.mask].data, 90, axis=0
                                )
                            windowrange = np.roll(windowrange, 1)
        else:
            print("Percentiles are retrieved from the thfile provided (PA13)")
            # A percentile file is provided and it contains a PRCTILE90 variable
            pct_file = nc.Dataset(thres_file, "r")
            pct_calc = pct_file.variables["PRCTILE90"][:].astype("float")
    else:
        raise ValueError("Method not supported: Choose between NF13 or PA13")
    
    return pct_calc


In [ ]:
def calc_spell(series):

    if isinstance(series, np.ma.core.MaskedArray):
        if np.any(series.mask == True):
            series[series.mask] = -99

    srun = np.zeros(series.shape)
    srun[1:] = np.diff(series, axis=0)
    srun[srun == 99] = -1
    srun[srun == 100] = 1
    srun[0] = -1
    if isinstance(series, np.ma.core.MaskedArray):
        L = (series.data).tolist()
    else:
        L = (series).tolist()
    groups_hw = []

    for k, g in groupby(L):
        if k == 1:
            b = list(g)
            groups_hw.append(sum(b))

    spell_hw = np.zeros((len(series),), dtype=int)
    if np.any(srun == 1):
        spell_hw[srun == 1] = np.asarray(groups_hw)

    ## Keep only spells equal or larger than 3 days

    spell_hw[spell_hw < 3] = 0
    return spell_hw

In [ ]:
def tave_tm_bnds(mask,dates,tave):

    if mask == None:
        mask = np.ones(tave.shape[1:], int)

    # PERFORM SOME CHECKS
    ## This is explicitly checked to preserve compatibility across versions
    if (bsyear == None) or (beyear == None):
        sys.exit(
            "ERROR: you didn't provide base period years to compute_EHF function, please revise"
        )
        
    years_all = np.asarray([dates[i].year for i in range(len(dates))])
    months_all = np.asarray([dates[i].month for i in range(len(dates))])
    days_all = np.asarray([dates[i].day for i in range(len(dates))])

    # If using PA13, leap days need to be removed

    if method == "PA13":

        dates = dates[((months_all == 2) & (days_all == 29)) == False]
        years = np.asarray([dates[i].year for i in range(len(dates))])
        months = np.asarray([dates[i].month for i in range(len(dates))])
        days = np.asarray([dates[i].day for i in range(len(dates))])

        tave = tave[((months_all == 2) & (days_all == 29)) == False, :, :]

    else:

        years = np.asarray([dates[i].year for i in range(len(dates))])
        months = np.asarray([dates[i].month for i in range(len(dates))])
        days = np.asarray([dates[i].day for i in range(len(dates))])

    # Specify when the year start
    # It is important to define seasons (e.g. Souther Hemisphere, month_starty should be in winter)
    new_years = years.copy()
    new_years[months < month_starty] -= 1

    syear = np.min(years)
    eyear = np.max(years)
    nyears = eyear - syear + 1
    nbyears = beyear - bsyear + 1

    shift_pct = np.argmax(new_years == syear)

    ndays = tave.shape[0]
    nlat = tave.shape[1]
    nlon = tave.shape[2]

    return tave, nbyears, years

In [ ]:
def calc_EHF():
    """Calculate EHF, exceedance flag, and 3‑day mean temperature.

    This mirrors the logic in compute_EHFheatwaves.py but operates on
    Dask-backed xarray DataArrays where possible.
    """
    # 3‑day mean centred on the current day (uses last 3 days)
    tave_3days = tave.rolling(time=3).mean().fillna(0)

    # 30‑day mean using the window [t-32, t-3] (30 days) as in Argueso code:
    # implemented via a shift of 3 days then a 30‑day rolling mean.
    if EHFaccl:
        tave_30days = (
            tave.shift(time=3)
            .rolling(time=30, min_periods=30)
            .mean()
            .fillna(0)
        )
    else:
        tave_30days = None

    ###############################################
    ###############################################
    # CALCULATING EHIsig and EHIaccl (if required)

    ndays = tave.sizes["time"]

    if method == "PA13":
        # For PA13 we must use day-of-year-dependent percentiles.
        EHIsig = np.zeros(tave.shape, dtype=float)
        for t_idx in range(ndays):
            EHIsig[t_idx, :, :] = tave_3days.isel(time=t_idx) - pct[t_idx % 365, :, :]
        EHIsig = xr.DataArray(EHIsig, coords=tave_3days.coords, dims=tave_3days.dims)
    else:
        # NF13: pct is time‑independent (lat, lon) and broadcasts naturally
        EHIsig = tave_3days - pct

    if EHFaccl:
        EHIaccl = tave_3days - tave_30days

    ###############################################
    ###############################################
    # CALCULATING EHF and EHF_exceed

    if EHFaccl:
        # Match original: EHF = max(1, EHIaccl) * EHIsig
        factor = xr.where(EHIaccl > 1.0, EHIaccl, 1.0)
        EHF = factor * EHIsig
    else:
        EHF = EHIsig

    # Zero out negative EHF values as in the reference code
    EHF = EHF.where(EHF > 0, 0)

    # Exceedance flag: 1 where EHF > 0, else 0
    EHF_exceed = xr.where(EHF > 0, 1, 0)

    # Apply seasonal masking (summer_sh, summer_nh, yearly) to match reference behaviour
    # Uses global "years" from tave_tm_bnds and "dates" from open_files.
    global years
    syear = int(years[years > -90].min()) if np.any(years > -90) else int(years.min())
    EHF_exceed, years = zero_days(EHF_exceed, dates, years, season, syear)

    return EHF, EHF_exceed, tave_3days

In [ ]:
def zero_days(EHF_exceed, dates, years, season, syear):
    """Zero out days not belonging to the analysis season.

    Replicates the seasonal masking in compute_EHFheatwaves.py:
    - For SH summer (summer_sh): keep NOV–MAR, zero APR–OCT.
    - For NH summer (summer_nh): keep MAY–SEP, zero OCT–APR.
    - For yearly: keep all days.

    Parameters
    ----------
    EHF_exceed : xr.DataArray (time, lat, lon)
        Exceedance flag (0/1) before seasonal masking.
    dates : pandas.DatetimeIndex or array-like of datetime64
        Daily dates corresponding to the time axis.
    years : np.ndarray[int]
        Year labels per day (will be updated with -99 for masked days).
    season : {"summer_sh", "summer_nh", "yearly"}
        Season definition.
    syear : int
        Start year (used only to compute shift_start_year, for compatibility).
    """
    months = np.array([d.month for d in dates])

    if season == "summer_sh":
        # SH summer: keep NOV–MAR; zero APR–OCT
        mask_out = (months >= 4) & (months <= 10)
        # Update years as in the reference code
        years[mask_out] = -99
        # Apply mask along time dimension on EHF_exceed
        time_mask = xr.DataArray(mask_out, dims=["time"], coords={"time": EHF_exceed.time})
        EHF_exceed = EHF_exceed.where(~time_mask, 0)
        # Kept for compatibility with original, though unused here
        shift_start_year = (dt.datetime(syear, 11, 1) - dt.datetime(syear, 7, 1)).days
        _ = shift_start_year
    elif season == "summer_nh":
        # NH summer: keep MAY–SEP; zero OCT–APR
        mask_out = (months >= 10) | (months <= 4)
        years[mask_out] = -99
        time_mask = xr.DataArray(mask_out, dims=["time"], coords={"time": EHF_exceed.time})
        EHF_exceed = EHF_exceed.where(~time_mask, 0)
    elif season == "yearly":
        # No masking
        pass
    else:
        raise ValueError("Season not supported: Choose between summer_sh, summer_nh or yearly")
    
    return EHF_exceed, years

In [ ]:
def compute_heatwave_metrics(EHF_exceed_1D, EHF_1D, TMP3D_1D, mask):
    ndays = EHF_1D.shape[0]
    spell = calc_spell(EHF_exceed_1D)

    # Initialize outputs
    avg = np.full_like(EHF_1D, const.missingval, dtype=float)
    peak = np.full_like(EHF_1D, const.missingval, dtype=float)
    t3d_peak = np.full_like(EHF_1D, const.missingval, dtype=float)
    t3d_avg = np.full_like(EHF_1D, const.missingval, dtype=float)
    ehf_flag = np.zeros_like(EHF_1D, dtype=int)

    if bool(mask):
        for t in range(ndays):
            if spell[t] != 0:
                span = spell[t]
                start = t
                end = t + span
                # Event-level characteristics taken over the full spell [t, t+span)
                avg[start] = np.mean(EHF_1D[start:end])
                peak[start] = np.max(EHF_1D[start:end])
                t3d_peak[start] = np.max(TMP3D_1D[start:end])
                t3d_avg[start] = np.mean(TMP3D_1D[start:end])
                # Flag all days in the spell as heatwave days
                ehf_flag[start:end] = EHF_exceed_1D[start:end]

    return avg, peak, t3d_peak, t3d_avg, ehf_flag

In [ ]:
def open_files(files):
    fin = xr.open_mfdataset(
        files,
        concat_dim='time',
        combine='nested',
        parallel=True,
        data_vars='minimal',
        coords='minimal',
        drop_variables="time_bnds",
        chunks="auto",
        engine='h5netcdf',
    )

    # hourly 2‑m temperature in UTC
    tas = fin.tas

    # convert nominal UTC timestamps to fixed AEST (UTC+10), no DST
    time_utc = pd.DatetimeIndex(tas.time.values)
    time_aest = time_utc + pd.Timedelta(hours=10)
    tas = tas.assign_coords(time=time_aest)

    # daily Tmax/Tmin from hourly AEST, then daily mean = (Tmax+Tmin)/2
    tas_daily_max = tas.resample(time='1D').max()
    tas_daily_min = tas.resample(time='1D').min()
    tave = ((tas_daily_max + tas_daily_min) / 2.0).rename("tas")
    tave = tave.chunk(time=-1, lat=100, lon=257)

    dates = pd.to_datetime(tave.time.values)

    return tave, dates

Here you should give the filepaths to a the threshold file, if you don't have it, it should work but I'm not sure how fast.

Also, a land-sea mask can be opened under mask. This will still work if you don't include a mask and will keep all the data points.

In [ ]:
thres_file="/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/t95_baseline.nc"
bsyear=1979
beyear=2000

month_starty=7
EHFaccl=True
method="NF13"
mask = xr.open_dataarray("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/land_sea_mask.nc").astype(bool)
season="yearly"

The Argueso code also includes calculations for HWF, HWA, etc. It wasn't relevant to me so I removed it.

In [ ]:
sdate, edate ='20190701', '20240630'

fdates = [m.strftime('%Y%m') for m in pd.date_range(sdate, edate, freq='ME')]
fpaths = [s for s in os.listdir(tas_path) if any(f in s for f in fdates)]
fpaths = sorted([tas_path + f for f in fpaths])

tave, dates = open_files(fpaths)
tave, nbyears, years = tave_tm_bnds(True,dates,tave)

In [ ]:
tave

In [ ]:
tave.chunks

In [ ]:
pct = calc_percentile(
    tave[(years >= bsyear) & (years <= beyear), :, :],
    nbyears,
    thres_file,
    method=method,
    nwindow=15,
    )

EHF, EHF_exceed, tave_3days = calc_EHF()
heatwave_EHF_avg, heatwave_EHF_peak, heatwave_TMP3D_peak, heatwave_TMP3D_avg, heatwave_EHF = xr.apply_ufunc(
                                                                                                        compute_heatwave_metrics,
                                                                                                        EHF_exceed,
                                                                                                        EHF,
                                                                                                        tave_3days,
                                                                                                        mask,
                                                                                                        input_core_dims=[['time'], ['time'], ['time'],[]],
                                                                                                        output_core_dims=[['time'], ['time'], ['time'], ['time'], ['time']],
                                                                                                        vectorize=True,
                                                                                                        dask='parallelized',
                                                                                                        output_dtypes=[float, float, float, float, float]
                                                                                                        )




In [ ]:
HW_EHF = tave.to_dataset().transpose('lat','lon','time').assign(EHF_flag = heatwave_EHF,
                                  EHF_val = EHF.transpose('lat','lon','time'),
                                  HW_EHF_avg = heatwave_EHF_avg,
                                  HW_EHF_peak = heatwave_EHF_peak,
                                  tas_3d_avg = heatwave_TMP3D_avg,
                                  tas_3d_peak = heatwave_TMP3D_peak)


In [ ]:
HW_EHF

In [ ]:
HW_EHF['tas'].chunks

In [ ]:
start_julys = pd.date_range(start=dates[0], end=dates[-1], freq='12MS')

encoding = {v: {"zlib": True, "complevel": 4, "shuffle": True} for v in HW_EHF.data_vars}

for start in start_julys:
    end = start + pd.DateOffset(months=12) - pd.DateOffset(days=1)
    chunk = HW_EHF.sel(time=slice(start, end)).compute()
    if chunk.time.size > 0:
        chunk.to_netcdf(f"{write_path}HW_EHF_{start.year}_{(start + pd.DateOffset(months=11)).year}.nc",
                       encoding=encoding,
                       compute=False,
                       engine='netcdf4')
    del chunk

In [ ]:
HW_EHF